In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_10_classes', 'seed': 42, 'n_classes': 10}, 'dataset': {'split_type': 'temporal', 'test_split': 0.2, 'cutoff_year': 1996}, 'paths': {'data_exploration_dir': 'output/experiment_with_10_classes/data_exploration', 'artifacts_dir': 'output/experiment_with_10_classes/artifacts', 'embeddings_dir': 'output/experiment_with_10_classes/embeddings', 'models_dir': 'output/experiment_with_10_classes/models', 'results_dir': 'output/experiment_with_10_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DAT

# Build Embeddings
This notebook generates embeddings for both SBERT (Sequence encoder) and OpenAI models and stores the indices inside the experiment folder.

In [2]:

import pathlib, json, numpy as np, faiss
from tqdm import tqdm
from src.datasets.dataset import get_dataset
from src.rag.vector_store import VectorStore
from src.rag import _ARTIFACTS_DIR, _SBERT_DIR, _OPENAI_DIR
from src.embeddings.openai_embedder import OpenAIEmbedder
from sentence_transformers import SentenceTransformer

# ---- Parameters ----
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_MODEL = "text-embedding-3-small"

print('Artifacts root:', _ARTIFACTS_DIR)
_SBERT_DIR.mkdir(parents=True, exist_ok=True)
_OPENAI_DIR.mkdir(parents=True, exist_ok=True)


INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artifacts root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/artifacts


In [3]:

# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: temporal
INFO |   - Number of classes: 10
INFO |   - Cutoff year: 1996
INFO |   - Random seed: None
INFO | Loading temporal split with cutoff year 1996
INFO | Temporal split: 0 documents before 1996, 0 from 1996 onward
/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/src/datasets/dataset.py:245: UserWarning: No test documents found in or after cutoff_year – falling back to default split.
  _warnings.warn("No test documents found in or after cutoff_year – falling back to default split.")
WARNING | No test documents found in or after 1996 - falling back to default split
INFO | Loading standard train/test split
INFO | Selected 10 classes: earn, acq, crude, interest, money-fx and more...
INFO | Dataset prepared with:
INFO |   - Training samples: 6337
INFO |   - Test samples: 2477
INFO |   - Classes: 10


Loaded 6337 training documents with 10 classes


In [4]:

# ---- SBERT embeddings ----
sbert = SentenceTransformer(SBERT_MODEL)
vectors = sbert.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')

meta = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, vectors)):
    meta.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

faiss_path = _SBERT_DIR / "index.faiss"
meta_path  = _SBERT_DIR / "meta.jsonl"
VectorStore.build(vectors, meta, vectors.shape[1], faiss_path, meta_path)
print("✅ SBERT index saved at", faiss_path)


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Batches: 100%|██████████| 100/100 [03:10<00:00,  1.91s/it]
INFO | Building FAISS index with 6337 documents of dimension 384
INFO | Creating IndexFlatIP...
INFO | Normalizing embeddings...
INFO | Adding embeddings to index...
INFO | Added embeddings to index in 0.00 seconds
INFO | Writing index to /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/embeddings/sbert/index.faiss...
INFO | Wrote index in 0.01 seconds
INFO | Writing metadata to /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/embeddings/sbert/meta.jsonl...
INFO | Wrote metadata in 1.97 seconds
INFO | Index built in 1.99 seconds


✅ SBERT index saved at /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/embeddings/sbert/index.faiss


In [5]:

# ---- OpenAI embeddings ----
# Requires OPENAI_API_KEY env var
openai_embedder = OpenAIEmbedder(model=OPENAI_MODEL, batch_size=50)
openai_vecs = openai_embedder.encode(X_train)
openai_vecs = np.array(openai_vecs, dtype='float32')

meta_openai = []
for i, (txt, label, vec) in enumerate(zip(X_train, y_train, openai_vecs)):
    meta_openai.append({"id": i, "label": label, "text": txt, "vector": vec.tolist()})

openai_faiss = _OPENAI_DIR / "index.faiss"
openai_meta  = _OPENAI_DIR / "meta.jsonl"
VectorStore.build(openai_vecs, meta_openai, openai_vecs.shape[1], openai_faiss, openai_meta)
print("✅ OpenAI index saved at", openai_faiss)


INFO | Starting OpenAI embedding generation for 6337 texts with model text-embedding-3-small
INFO | Processing embedding batch 1 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 1 used 8422 tokens
INFO | Batch 1 completed in 1.97 seconds
INFO | Average time per text in batch: 0.0394 seconds
INFO | Adding delay of 0.39s before next batch
INFO | Processing embedding batch 2 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 2 used 7517 tokens
INFO | Batch 2 completed in 1.97 seconds
INFO | Average time per text in batch: 0.0394 seconds
INFO | Adding delay of 0.21s before next batch
INFO | Processing embedding batch 3 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 3 used 10559 tokens
INFO | Batch 3 completed in 2.12 seconds
INFO | Average time per text in batch: 0.0423 seconds
INFO | Adding delay of 0.28s before ne

✅ OpenAI index saved at /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_10_classes/embeddings/openai/index.faiss
